# DQA Noise Study: Accuracy vs System Size

Compares **ideal** (noiseless statevector) vs **noisy** (depolarizing noise, trajectory simulation)
DQA expected values for `n_y ∈ {4, 6, 8, 10}` wind turbines on the **CUDA-Q nvidia** (H100) target.

For each problem size the workflow is:
1. Solve classically to obtain the reference optimal cost.
2. COBYLA-optimize DQA angles against the **ideal** statevector cost.
3. Evaluate the optimized circuit under (a) ideal SV and (b) depolarizing noise.
4. Record `|φ_est - φ_classical| / φ_classical` as the relative error.

Noise model: single-qubit depolarizing `p1`, two-qubit depolarizing `p2`.

In [ ]:
import os, sys, math, time
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

_QISKIT_SP   = '/nopt/nrel/apps/gpu_stack/software/qiskit/aer-gpu/venv/lib/python3.11/site-packages'
_QISKIT_IMPL = '/projects/hpcapps/nsawant/quantum_stochastic_programming/qiskit_impl'
for _p in [_QISKIT_SP, _QISKIT_IMPL]:
    if _p not in sys.path:
        sys.path.insert(0, _p)
os.chdir(_QISKIT_IMPL)

import cudaq
from cudaq_impl import CudaqQAEOptimizer, build_depolarizing_noise_model
from binary_optimizer import BinaryNestedOptimizer

%load_ext autoreload
%autoreload 2

cudaq.set_target('nvidia')
print('CUDA-Q target:', cudaq.get_target().name)
print('GPUs available:', cudaq.num_available_gpus())
print('Imports OK')

In [ ]:
# ── NOISE MODEL PARAMETERS ────────────────────────────────────────────────────
# Typical near-term H100-class gate fidelities:
#   p1 = 0.001  (1-qubit depolarizing)
#   p2 = 0.01   (2-qubit depolarizing)
P1 = 0.001    # single-qubit depolarizing probability
P2 = 0.010    # two-qubit depolarizing probability
NUM_TRAJECTORIES = 2048  # Monte-Carlo trajectories for noisy observe()

noise_model = build_depolarizing_noise_model(p1=P1, p2=P2)
print(f'Noise model built: p1={P1}, p2={P2}')
print(f'Trajectories per run: {NUM_TRAJECTORIES}')

In [ ]:
# ── PROBLEM CONFIGURATION FOR EACH n_y ───────────────────────────────────────
# c_y values scale with system size (each turbine adds a new cost tier).
# w_d = floor(n_y / 2) — demand met by roughly half the turbines.
_BASE_CY = [0.4, 0.5, 0.6, 0.7, 0.8, 1.0, 1.2, 1.4, 1.6, 1.8]

N_Y_LIST = [4, 6, 8, 10]

CONFIGS = {}
for ny in N_Y_LIST:
    w = ny // 2
    cy = _BASE_CY[:ny]
    cr = 10.
    cost_norm = w * cr / ny
    norm      = w * cr
    ts        = ny                  # ansatz depth = n_y timesteps
    theta0    = []
    for t in range(ts):
        theta0.append(float(t / ts))
        theta0.append((1 - float(t / ts)) / math.pi)
    CONFIGS[ny] = dict(
        n_y=ny, w_d=w, c_y=cy, c_r=cr,
        cost_norm=cost_norm, norm=norm,
        timesteps=ts, theta0=theta0,
    )
    print(f"n_y={ny:2d}  w_d={w}  cost_norm={cost_norm:.2f}  norm={norm}  "
          f"angles={len(theta0)}  c_y={cy}")

In [ ]:
# ── CLASSICAL REFERENCE SOLUTIONS ────────────────────────────────────────────
# Brute-force exact second-stage cost for each n_y.
classical_phi = {}

for ny, cfg in CONFIGS.items():
    cy, cr, w = cfg['c_y'], cfg['c_r'], cfg['w_d']
    # Enumerate all 2^ny scenarios; wind demand = w out of ny turbines.
    # Scenario xi: binary vector of length ny (1 = wind available)
    # x (first-stage): commit exactly w turbines  (y = indicator of commitment)
    # Optimal y = commit cheapest w turbines (indices sorted by c_y).
    # Then E[Q(y*, xi)] over uniform xi.
    sorted_idx = sorted(range(ny), key=lambda j: cy[j])
    y_star = [0] * ny
    for j in sorted_idx[:w]:
        y_star[j] = 1

    total_cost = 0.0
    n_scenarios = 2 ** ny
    for xi_int in range(n_scenarios):
        xi = [(xi_int >> j) & 1 for j in range(ny)]
        cost = sum(
            (cy[j] if xi[j] == 1 else cr) for j in range(ny) if y_star[j] == 1
        )
        total_cost += cost
    classical_phi[ny] = total_cost / n_scenarios
    print(f"n_y={ny}  classical φ* = {classical_phi[ny]:.4f}")

In [ ]:
# ── COBYLA OPTIMIZATION (ideal SV) ───────────────────────────────────────────
# For each n_y, optimise DQA angles against the exact (noiseless) SV cost.
# This may take a few minutes for larger n_y.

optimized_thetas = {}
cobyla_results   = {}

for ny, cfg in CONFIGS.items():
    print(f"\n── Optimising n_y={ny} ────────────────────────────────")
    opt_ideal = CudaqQAEOptimizer(
        c_x=[3.], c_y=cfg['c_y'], c_r=cfg['c_r'],
        n_y=ny, w_d=cfg['w_d'], cost_norm=cfg['cost_norm'],
        noise_model=None,          # ideal for optimisation
    )

    call_count = [0]
    def _obj(th):
        call_count[0] += 1
        return opt_ideal.estimate_expected_value_sv(th.tolist(), cfg['w_d'])

    t0 = time.perf_counter()
    result = minimize(
        _obj, cfg['theta0'],
        method='COBYLA',
        options={'maxiter': 500, 'rhobeg': 0.5, 'catol': 1e-4},
    )
    dt = time.perf_counter() - t0

    optimized_thetas[ny] = result.x.tolist()
    cobyla_results[ny]   = result
    phi_opt = result.fun
    phi_ref = classical_phi[ny]
    err_pct = abs(phi_opt - phi_ref) / phi_ref * 100
    print(f"  φ_DQA_ideal = {phi_opt:.4f}  |  φ_classical = {phi_ref:.4f}")
    print(f"  error = {err_pct:.2f}%  |  evals={call_count[0]}  |  time={dt:.1f}s")

In [ ]:
# ── NOISY EVALUATION WITH OPTIMISED ANGLES ────────────────────────────────────
# Re-evaluate the COBYLA-optimised angles under the depolarizing noise model.
# Uses cudaq.observe() + num_trajectories Monte-Carlo trajectories on the GPU.

results_ideal = {}   # {ny: phi}
results_noisy = {}   # {ny: phi}

for ny, thetas in optimized_thetas.items():
    cfg = CONFIGS[ny]
    print(f"\n── Evaluating n_y={ny} ────────────────────────────────")

    # --- ideal (exact statevector) ---
    opt_ideal = CudaqQAEOptimizer(
        c_x=[3.], c_y=cfg['c_y'], c_r=cfg['c_r'],
        n_y=ny, w_d=cfg['w_d'], cost_norm=cfg['cost_norm'],
        noise_model=None,
    )
    t0 = time.perf_counter()
    phi_ideal = opt_ideal.estimate_expected_value_sv(thetas, cfg['w_d'])
    dt_ideal  = time.perf_counter() - t0
    results_ideal[ny] = phi_ideal

    # --- noisy (trajectory simulation) ---
    opt_noisy = CudaqQAEOptimizer(
        c_x=[3.], c_y=cfg['c_y'], c_r=cfg['c_r'],
        n_y=ny, w_d=cfg['w_d'], cost_norm=cfg['cost_norm'],
        noise_model=noise_model,
    )
    t0 = time.perf_counter()
    phi_noisy = opt_noisy.estimate_expected_value_noisy_trajectory(
                    thetas, num_trajectories=NUM_TRAJECTORIES)
    dt_noisy  = time.perf_counter() - t0
    results_noisy[ny] = phi_noisy

    ref = classical_phi[ny]
    err_i = abs(phi_ideal - ref) / ref * 100
    err_n = abs(phi_noisy - ref) / ref * 100
    print(f"  Ideal : φ={phi_ideal:.4f}  error={err_i:.2f}%  ({dt_ideal*1e3:.0f} ms)")
    print(f"  Noisy : φ={phi_noisy:.4f}  error={err_n:.2f}%  ({dt_noisy*1e3:.0f} ms)")

In [ ]:
# ── PLOT: ACCURACY vs SYSTEM SIZE ─────────────────────────────────────────────

ny_vals   = N_Y_LIST
ref_vals  = [classical_phi[ny] for ny in ny_vals]
err_ideal = [abs(results_ideal[ny] - classical_phi[ny]) / classical_phi[ny] * 100
             for ny in ny_vals]
err_noisy = [abs(results_noisy[ny] - classical_phi[ny]) / classical_phi[ny] * 100
             for ny in ny_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- left panel: absolute φ values ---
ax = axes[0]
ax.plot(ny_vals, ref_vals,        'k--o',  lw=2, ms=8, label='Classical optimal')
ax.plot(ny_vals, [results_ideal[n] for n in ny_vals],
        'b-s',  lw=2, ms=8, label='DQA ideal (noiseless SV)')
ax.plot(ny_vals, [results_noisy[n] for n in ny_vals],
        'r-^',  lw=2, ms=8,
        label=f'DQA noisy (p₁={P1}, p₂={P2}, N={NUM_TRAJECTORIES} traj.)')
ax.set_xlabel('Number of turbines $n_y$', fontsize=12)
ax.set_ylabel('Expected second-stage cost  $\\phi$', fontsize=12)
ax.set_title('Expected cost vs system size', fontsize=13)
ax.set_xticks(ny_vals)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)

# --- right panel: relative error (%) ---
ax = axes[1]
x   = np.arange(len(ny_vals))
w   = 0.32
b1  = ax.bar(x - w/2, err_ideal, width=w, color='steelblue', alpha=0.85,
             label='Ideal (noiseless)')
b2  = ax.bar(x + w/2, err_noisy, width=w, color='tomato',    alpha=0.85,
             label=f'Noisy (p₁={P1}, p₂={P2})')
for rect, val in zip(list(b1) + list(b2), err_ideal + err_noisy):
    ax.text(rect.get_x() + rect.get_width() / 2, rect.get_height() + 0.1,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
ax.set_xlabel('Number of turbines $n_y$', fontsize=12)
ax.set_ylabel('Relative error  (%)  $= |\\phi_{\\rm DQA} - \\phi^*| / \\phi^*$', fontsize=11)
ax.set_title(f'DQA accuracy: ideal vs noisy\n(depolarizing, {NUM_TRAJECTORIES} trajectories)', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels([f'$n_y={n}$' for n in ny_vals])
ax.legend(fontsize=10)
ax.grid(True, axis='y', alpha=0.4)

plt.tight_layout()
plt.savefig('noise_study_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to noise_study_accuracy.png')

In [ ]:
# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
print(f"{'n_y':>4} {'φ_classical':>12} {'φ_ideal':>10} "
      f"{'err_ideal(%)':>13} {'φ_noisy':>10} {'err_noisy(%)':>13}")
print('-' * 65)
for ny in ny_vals:
    ref = classical_phi[ny]
    pi  = results_ideal[ny]
    pn  = results_noisy[ny]
    ei  = abs(pi - ref) / ref * 100
    en  = abs(pn - ref) / ref * 100
    print(f"{ny:>4} {ref:>12.4f} {pi:>10.4f} {ei:>13.2f} {pn:>10.4f} {en:>13.2f}")